# Portfolio optimization
*Algorithms for Optimization, Master Program in Finance, University of Pavia. Notebook of Lecture 6.*

| Section | Topic |
|---|---|
| 0 | Setup |
| 1 | Data: returns, expected returns, covariances |
| 2 | The Markowitz model |
| 3 | The efficient frontier |
| 4 | The MAD model (an LP) |
| 5 | Cardinality constraints (a MIQP) |
| 6 | Exercise: maximum return for a given risk |
| 7 | In-sample vs out-of-sample |
| 8 | Optional: real data |

Exercises are marked with ✏️.

## 0. Setup

In [ ]:
!pip install gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/AmbrogioMB/AlgOpt/main/data/"

## 1. Data: returns, expected returns, covariances
The file `asset_returns.csv` contains 10 years (120 months) of monthly returns of 10 assets; `assets.csv` contains their sector. The data are **simulated** and the tickers are fictional, as in the first part of the course.

In [ ]:
returns = pd.read_csv(DATA_URL + "asset_returns.csv", parse_dates=["date"], index_col="date")
assets = pd.read_csv(DATA_URL + "assets.csv")
returns.head()

Remember the third source of error of the introduction: if the assets appear in different orders in different objects, the code runs without errors and computes the wrong portfolio. We **check** that the columns of `returns` and the rows of `assets` are in the same order.

In [ ]:
assert list(returns.columns) == list(assets["ticker"]), "different order of the assets!"
tickers = list(returns.columns)
sector = list(assets["sector"])
n = len(tickers)
S = len(returns)
print(f"{n} assets, {S} months")

We estimate the expected returns $\mu$ and the covariance matrix $\Sigma$ from the data and **annualize** them (monthly mean $\times 12$, monthly covariance $\times 12$), so that all the numbers below are per year.

In [ ]:
mu = 12 * returns.mean().to_numpy()
Sigma = 12 * returns.cov().to_numpy()
vol = np.sqrt(np.diag(Sigma))

summary = pd.DataFrame({"sector": sector, "exp. return": mu, "volatility": vol}, index=tickers)
print(summary.round(4))
print("\nCorrelations:")
print(returns.corr().round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
(1 + returns).cumprod().plot(ax=ax[0], legend=False)
ax[0].set_title("Value of 1 euro invested in each asset")
ax[0].set_ylabel("value")
ax[1].scatter(vol, mu)
for j in range(n):
    ax[1].annotate(tickers[j], (vol[j], mu[j]), textcoords="offset points", xytext=(5, 3))
ax[1].set_xlabel("volatility (annual)")
ax[1].set_ylabel("expected return (annual)")
ax[1].set_title("Risk and return of the single assets")
ax[1].grid(True)
plt.show()

## 2. The Markowitz model
For a target return $R$ we solve the quadratic program
$$
\min_w \; w^T \Sigma w \quad \text{s.t.} \quad \mu^T w \geq R, \quad \textstyle\sum_j w_j = 1, \quad w \geq 0.
$$
With the matrix interface of `gurobipy` (`addMVar`) the objective $w^T \Sigma w$ is simply `w @ Sigma @ w`. We write a function, because we will solve the model many times.

In [ ]:
def markowitz(R, verbose=False):
    m = gp.Model("Markowitz")
    m.Params.OutputFlag = 1 if verbose else 0
    w = m.addMVar(n, lb=0, name="w")
    m.setObjective(w @ Sigma @ w, GRB.MINIMIZE)
    m.addConstr(mu @ w >= R, name="target")
    m.addConstr(w.sum() == 1, name="budget")
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        return None, None
    return w.X, m.ObjVal


def show(w):
    for j in range(n):
        if w[j] > 1e-4:
            print(f"  {tickers[j]:8s} {sector[j]:12s} {100 * w[j]:6.2f}%")
    print(f"  expected return {100 * mu @ w:.2f}%, volatility {100 * np.sqrt(w @ Sigma @ w):.2f}%")


w_opt, var_opt = markowitz(0.10, verbose=True)
print()
show(w_opt)

## 3. The efficient frontier
We solve the Markowitz model for many targets $R$, from the return of the **minimum variance portfolio** (no target) to the largest expected return.

In [ ]:
w_min, _ = markowitz(-1)   # a target of -100% is always satisfied: minimum variance portfolio
print("Minimum variance portfolio:")
show(w_min)

targets = np.linspace(mu @ w_min, mu.max(), 30)
frontier_w = [markowitz(R)[0] for R in targets]
frontier_risk = [np.sqrt(w @ Sigma @ w) for w in frontier_w]
frontier_ret = [mu @ w for w in frontier_w]

plt.figure(figsize=(8, 5))
plt.plot(frontier_risk, frontier_ret, "r-", linewidth=2, label="efficient frontier")
plt.scatter(vol, mu, label="single assets")
for j in range(n):
    plt.annotate(tickers[j], (vol[j], mu[j]), textcoords="offset points", xytext=(5, 3))
plt.xlabel("volatility (annual)")
plt.ylabel("expected return (annual)")
plt.legend()
plt.grid(True)
plt.show()

Diversification pays off: the frontier lies to the left of the single assets. How does the composition change along the frontier?

In [ ]:
plt.figure(figsize=(8, 5))
plt.stackplot(frontier_ret, np.array(frontier_w).T, labels=tickers)
plt.xlabel("expected return (annual)")
plt.ylabel("weight")
plt.legend(loc="upper left", fontsize=8, ncol=2)
plt.show()

## 4. The MAD model (an LP)
Instead of the variance, we measure the risk with the **mean absolute deviation** of the monthly portfolio return over the $S$ observed months:
$$
\text{MAD}(w) = \frac{1}{S} \sum_{s=1}^S \Big| \sum_{j=1}^n (r_{sj} - \bar r_j) w_j \Big|,
$$
where $\bar r_j$ is the average monthly return of asset $j$. With the absolute value trick of Lecture 4 we obtain a **linear program**:
$$
\min_{w, d} \; \frac{1}{S} \sum_{s=1}^S d_s \quad \text{s.t.} \quad d_s \geq \textstyle\sum_{j} (r_{sj} - \bar r_j) w_j, \quad d_s \geq - \textstyle\sum_{j} (r_{sj} - \bar r_j) w_j, \quad \mu^T w \geq R, \quad \textstyle\sum_{j} w_j = 1, \quad w \geq 0.
$$

In [ ]:
deviations = (returns - returns.mean()).to_numpy()   # S x n matrix


def mad(R):
    m = gp.Model("MAD")
    m.Params.OutputFlag = 0
    w = m.addMVar(n, lb=0, name="w")
    d = m.addMVar(S, lb=0, name="d")
    m.setObjective(d.sum() / S, GRB.MINIMIZE)
    m.addConstr(d >= deviations @ w, name="abs_pos")
    m.addConstr(d >= -(deviations @ w), name="abs_neg")
    m.addConstr(mu @ w >= R, name="target")
    m.addConstr(w.sum() == 1, name="budget")
    m.optimize()
    return w.X, m.ObjVal


w_mad, _ = mad(0.10)
w_mv, _ = markowitz(0.10)
print(pd.DataFrame({"Markowitz": 100 * w_mv, "MAD": 100 * w_mad}, index=tickers).round(2))
print(f"\nvolatility of the Markowitz portfolio: {100 * np.sqrt(w_mv @ Sigma @ w_mv):.2f}%")
print(f"volatility of the MAD portfolio:       {100 * np.sqrt(w_mad @ Sigma @ w_mad):.2f}%")

The two portfolios are similar, and the MAD portfolio is almost as good as the Markowitz one even when measured with the volatility. The MAD model is an LP: no covariance matrix, and all the LP tools (duality, sensitivity analysis) are available.

## 5. Cardinality constraints (a MIQP)
Suppose we want to hold **at most $K = 3$ assets**, and each asset we hold must have a weight of at least $\ell = 10\%$. We introduce binary variables $\delta_j$ (module 4 of the course):
$$
\ell \, \delta_j \leq w_j \leq \delta_j, \qquad \sum_j \delta_j \leq K, \qquad \delta_j \in \{0, 1\}.
$$
The problem becomes a **mixed integer quadratic program** (MIQP), solved with branch-and-bound. Look at the log.

In [ ]:
def markowitz_card(R, K, ell, verbose=False):
    m = gp.Model("Markowitz with cardinality")
    m.Params.OutputFlag = 1 if verbose else 0
    w = m.addMVar(n, lb=0, name="w")
    delta = m.addMVar(n, vtype=GRB.BINARY, name="delta")
    m.setObjective(w @ Sigma @ w, GRB.MINIMIZE)
    m.addConstr(mu @ w >= R, name="target")
    m.addConstr(w.sum() == 1, name="budget")
    m.addConstr(w <= delta, name="link")
    m.addConstr(w >= ell * delta, name="min_position")
    m.addConstr(delta.sum() <= K, name="cardinality")
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        return None, None
    return w.X, m.ObjVal


w_card, var_card = markowitz_card(0.10, K=3, ell=0.10, verbose=True)
print("\nWith at most 3 assets:")
show(w_card)
print("Without cardinality constraint:")
show(w_mv)

Linear constraints such as **sector limits** are added in one line. For example, at most 30% in the Tech sector:

In [ ]:
tech = np.array([s == "Tech" for s in sector])
print("Tech weight in the K = 3 portfolio:", round(w_card[tech].sum(), 3))
# inside markowitz_card one would add:  m.addConstr(w[tech].sum() <= 0.30, name="tech_limit")

### Exercise 5.1 ✏️
Plot on the same figure the efficient frontier without cardinality constraint and with $K = 2$ and $K = 3$ (with $\ell = 10\%$). What do you observe? How large is the price of holding only 2 or 3 assets?

In [ ]:
# YOUR CODE HERE

## 6. Exercise ✏️: maximum return for a given risk (Exercise 2 of Lecture 6)
Write a variant of the Markowitz model that **maximizes the expected return** subject to a maximum volatility of $\sigma_{\max} = 12\%$ per year, i.e., the quadratic constraint $w^T \Sigma w \leq \sigma_{\max}^2$. Gurobi accepts quadratic constraints written like linear ones. Is the portfolio on the efficient frontier?

In [ ]:
# YOUR CODE HERE
sigma_max = 0.12

## 7. In-sample vs out-of-sample
The model trusts the estimates of $\mu$ and $\Sigma$, but they are noisy. A fair test: **estimate** on the first 5 years (2016–2020, *in-sample*) and **evaluate** the portfolios on the last 5 years (2021–2025, *out-of-sample*), which the model has never seen.

In [ ]:
train = returns.loc[:"2020-12-31"]
test = returns.loc["2021-01-01":]
mu_train = 12 * train.mean().to_numpy()
Sigma_train = 12 * train.cov().to_numpy()


def realized(w, data):
    """Annual return and volatility of the portfolio w on the monthly returns in data."""
    r = data.to_numpy() @ w
    return 12 * r.mean(), np.sqrt(12) * r.std(ddof=1)


print(len(train), "months in-sample,", len(test), "months out-of-sample")

### Exercise 7.1 ✏️
Using only `mu_train` and `Sigma_train`, compute
1. the minimum variance portfolio,
2. the Markowitz portfolio with target return 12% (in-sample),
3. the equally weighted portfolio $w_j = 1/n$.

For each one, compare the expected return and volatility predicted in-sample with the realized ones out-of-sample (function `realized`). Which portfolio keeps its promises? Compare also the in-sample and out-of-sample expected returns of the single assets.

*Hint:* the function `markowitz` uses the global `mu` and `Sigma`; write a version `markowitz_est(R, mu_hat, Sigma_hat)` that takes them as arguments.

In [ ]:
# YOUR CODE HERE

## 8. Optional: real data
You can download real prices with the `yfinance` package and compute monthly returns. Uncomment the cell, run it, and then run Sections 1–6 again **skipping the cell that reads the csv files** (the sector is set to "n/a"). Data from Yahoo Finance are free but not always reliable: always inspect them first.

In [ ]:
# !pip install yfinance
# import yfinance as yf
# tickers = ["ISP.MI", "UCG.MI", "ENI.MI", "STM.MI", "RACE.MI", "GLD", "IEF"]
# prices = yf.download(tickers, start="2016-01-01", end="2026-01-01", interval="1mo", auto_adjust=True)["Close"]
# returns = prices[tickers].pct_change().dropna()
# tickers, sector = list(returns.columns), ["n/a"] * len(returns.columns)
# n, S = len(tickers), len(returns)
# mu, Sigma = 12 * returns.mean().to_numpy(), 12 * returns.cov().to_numpy()
# vol = np.sqrt(np.diag(Sigma))